# Assignment 5 — Word2Vec

**Resources:**
- Dataset: https://www.kaggle.com/datasets/kritanjalijain/amazon-reviews/data
- Model: https://huggingface.co/fse/word2vec-google-news-300
- Analogy file: https://github.com/nicholas-leonard/word2vec/blob/master/questions-words.txt

---
## Setup

In [ ]:
!pip install gensim nltk pandas numpy huggingface_hub -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 64.7 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from gensim.models import KeyedVectors
from huggingface_hub import hf_hub_download

nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')

print('All libraries imported!')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


All libraries imported!


## Load the Google Word2Vec Model

The HuggingFace repo (`fse/word2vec-google-news-300`) stores the model in **Gensim native format** — two files:
- `word2vec-google-news-300.model` — vocabulary and metadata (182 MB)
- `word2vec-google-news-300.model.vectors.npy` — the actual vectors (3.6 GB)

Both files must be downloaded. We load with `KeyedVectors.load()` (NOT `load_word2vec_format`).

In [ ]:
REPO = 'fse/word2vec-google-news-300'

print('Downloading .model file (182 MB)...')
model_path = hf_hub_download(repo_id=REPO, filename='word2vec-google-news-300.model')

print('Downloading .model.vectors.npy file (3.6 GB)...')
hf_hub_download(repo_id=REPO, filename='word2vec-google-news-300.model.vectors.npy')

print('Loading model...')
model = KeyedVectors.load(model_path)

print(f'Done! Vocab size: {len(model):,} words | Vector size: {model.vector_size}')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


word2vec-google-news-300.model:   0%|          | 0.00/182M [00:00<?, ?B/s]

word2vec-google-news-300.model.vectors.n(…):   0%|          | 0.00/3.60G [00:00<?, ?B/s]

Loading model...
Done! Vocab size: 3,000,000 words | Vector size: 300


---
## Q1 — Sentence Representation using Word Vectors

### Step 1: Load the Dataset

Download `train.csv` from Kaggle and place it in the same folder as this notebook.  
The CSV has no header row — columns are: `label`, `title`, `review`.

In [ ]:
import os
import kagglehub

# Download latest version
path = kagglehub.dataset_download("kritanjalijain/amazon-reviews")

print("Path to dataset files:", path)

100%|██████████| 1.29G/1.29G [00:16<00:00, 84.0MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/kritanjalijain/amazon-reviews/versions/2


In [ ]:
df = pd.read_csv(
    '/root/.cache/kagglehub/datasets/kritanjalijain/amazon-reviews/versions/2/train.csv',
    header=None,
    names=['label', 'title', 'review'],
    nrows=5000   # small sample to keep things fast
)

df = df.dropna(subset=['review'])  # remove any empty reviews

print(f'Loaded {len(df)} reviews')
print('\nSample reviews:')
df['review'].head(3)

Loaded 5000 reviews

Sample reviews:


,review
0,This sound track was beautiful! It paints the ...
1,I'm reading a lot of reviews saying that this ...
2,This soundtrack is my favorite music of all ti...


### Step 2: Preprocessing

We apply 4 steps to each review:
1. Convert to **lowercase**
2. Remove **punctuation and numbers**
3. **Tokenize** (split into individual words)
4. Remove **stopwords** (common words like "the", "is", "at" that carry no meaning)

In [ ]:
STOPWORDS = set(stopwords.words('english'))

def preprocess(text):
    # Step 1: lowercase
    text = text.lower()

    # Step 2: remove punctuation and numbers (keep only a-z and spaces)
    text = re.sub(r'[^a-z\s]', '', text)

    # Step 3: tokenize
    tokens = word_tokenize(text)

    # Step 4: remove stopwords
    tokens = [t for t in tokens if t not in STOPWORDS]

    return tokens


# --- Demo on one review ---
raw       = df['review'].iloc[0]
processed = preprocess(raw)

print('Raw review:')
print(raw[:200])
print('\nAfter preprocessing:')
print(processed[:20])

Raw review:
This sound track was beautiful! It paints the senery in your mind so well I would recomend it even to people who hate vid. game music! I have played the game Chrono Cross but out of all of the games I

After preprocessing:
['sound', 'track', 'beautiful', 'paints', 'senery', 'mind', 'well', 'would', 'recomend', 'even', 'people', 'hate', 'vid', 'game', 'music', 'played', 'game', 'chrono', 'cross', 'games']


In [ ]:
# Apply preprocessing to all reviews
df['tokens'] = df['review'].apply(preprocess)

print('Preprocessing done!')
df[['review', 'tokens']].head(3)

Preprocessing done!


,review,tokens
0,This sound track was beautiful! It paints the ...,"[sound, track, beautiful, paints, senery, mind..."
1,I'm reading a lot of reviews saying that this ...,"[im, reading, lot, reviews, saying, best, game..."
2,This soundtrack is my favorite music of all ti...,"[soundtrack, favorite, music, time, hands, int..."


### Step 3: Sentence Representation

Each word has a 300-dimensional vector in Word2Vec.  
To get a single vector for a whole sentence, we take the **average of all word vectors** in that sentence.

Words not found in the model vocabulary are simply skipped.

In [ ]:
def sentence_vector(tokens):
    """
    Returns the mean Word2Vec vector for a list of tokens.
    Words not in the vocabulary are ignored.
    If no words are found at all, returns a zero vector.
    """
    vectors = [model[word] for word in tokens if word in model]

    if len(vectors) == 0:
        return np.zeros(model.vector_size)

    return np.mean(vectors, axis=0)  # average all word vectors


# Apply to all reviews
df['vector'] = df['tokens'].apply(sentence_vector)

print('Sentence vectors computed!')
print(f'\nFirst review vector (first 10 of 300 dimensions):')
print(df['vector'].iloc[0][:10])
print(f'\nVector shape: {df["vector"].iloc[0].shape}')

Sentence vectors computed!

First review vector (first 10 of 300 dimensions):
[ 0.03934733  0.03986143  0.01863744  0.06304932 -0.02640631  0.04208922
  0.02736918 -0.0897209   0.02670992  0.06392337]

Vector shape: (300,)


In [ ]:
# --- Check: cosine similarity between two reviews ---
# Cosine similarity = 1 means identical direction, 0 means unrelated

from numpy.linalg import norm

def cosine_sim(v1, v2):
    return np.dot(v1, v2) / (norm(v1) * norm(v2))

v0 = df['vector'].iloc[0]
v1 = df['vector'].iloc[1]
v2 = df['vector'].iloc[2]

print('Review 0:', df['review'].iloc[0][:80])
print('Review 1:', df['review'].iloc[1][:80])
print('Review 2:', df['review'].iloc[2][:80])

print(f'\nSimilarity (Review 0 vs Review 1): {cosine_sim(v0, v1):.4f}')
print(f'Similarity (Review 0 vs Review 2): {cosine_sim(v0, v2):.4f}')

Review 0: This sound track was beautiful! It paints the senery in your mind so well I woul
Review 1: I'm reading a lot of reviews saying that this is the best 'game soundtrack' and 
Review 2: This soundtrack is my favorite music of all time, hands down. The intense sadnes

Similarity (Review 0 vs Review 1): 0.8033
Similarity (Review 0 vs Review 2): 0.8610


---
## Q2 — Word Analogies

### Concept

Word2Vec vectors capture semantic relationships. The analogy formula is:

```
vec(w2) - vec(w1) + vec(w3)  ≈  vec(answer)
```

Classic example: `king - man + woman = queen`

Gensim's `most_similar(positive=[w2, w3], negative=[w1])` does this in one line.

### Part a) good − bad + excellent = ?

In [ ]:
# good - bad + excellent = ?
# positive=[good, excellent], negative=[bad]

result = model.most_similar(positive=['good', 'excellent'], negative=['bad'], topn=5)

print('good - bad + excellent = ?')
print()
for rank, (word, score) in enumerate(result, 1):
    print(f'  {rank}. {word:<20}  score: {score:.4f}')

good - bad + excellent = ?

  1. terrific              score: 0.6986
  2. superb                score: 0.6440
  3. fantastic             score: 0.6410
  4. great                 score: 0.6320
  5. exceptional           score: 0.5893


### Part b) delivery − late + fast = ?

In [ ]:
# delivery - late + fast = ?
# positive=[delivery, fast], negative=[late]

result = model.most_similar(positive=['delivery', 'fast'], negative=['late'], topn=5)

print('delivery - late + fast = ?')
print()
for rank, (word, score) in enumerate(result, 1):
    print(f'  {rank}. {word:<20}  score: {score:.4f}')

delivery - late + fast = ?

  1. delievery             score: 0.4610
  2. twitch_muscles        score: 0.4286
  3. Delivery              score: 0.4249
  4. Eat##Hours.com        score: 0.3990
  5. www.BobBruss.com      score: 0.3976


### Part c) Evaluate on the official analogy dataset

**File:** `questions-words.txt`  
**Format:** Each line has 4 words: `w1  w2  w3  w4`  
**Task:** Given w1, w2, w3 → predict w4 using `vec(w2) - vec(w1) + vec(w3)`

#### How to minimize the search space?

The full vocabulary has 3 million words — searching all of them every time is very slow.

**Solution: `restrict_vocab` parameter**  
Pass `restrict_vocab=50000` to `most_similar()` — it then only searches the **top 50K most frequent words** instead of all 3M.  
This gives a ~60x speedup, and since most analogy answers are common words, accuracy barely changes.

In [ ]:
import urllib.request

# Download the analogy file
url  = 'https://raw.githubusercontent.com/nicholas-leonard/word2vec/master/questions-words.txt'
urllib.request.urlretrieve(url, 'questions-words.txt')
print('Downloaded questions-words.txt')

# Preview
with open('questions-words.txt') as f:
    lines = f.readlines()

print(f'Total lines: {len(lines)}')
print('\nFirst 8 lines:')
for line in lines[:8]:
    print(' ', line.strip())

Downloaded questions-words.txt
Total lines: 19558

First 8 lines:
  : capital-common-countries
  Athens Greece Baghdad Iraq
  Athens Greece Bangkok Thailand
  Athens Greece Beijing China
  Athens Greece Berlin Germany
  Athens Greece Bern Switzerland
  Athens Greece Cairo Egypt
  Athens Greece Canberra Australia


In [ ]:
# Parse into sections
# Lines starting with ':' are category headers

sections = {}
current  = None

for line in lines:
    line = line.strip()
    if line.startswith(':'):
        current = line[2:]           # section name
        sections[current] = []
    elif current and line:
        w1, w2, w3, w4 = line.lower().split()
        sections[current].append((w1, w2, w3, w4))

print(f'Total sections: {len(sections)}')
print()
for name, samples in sections.items():
    print(f'  {name:<40}  {len(samples)} samples')

Total sections: 14

  capital-common-countries                  506 samples
  capital-world                             4524 samples
  currency                                  866 samples
  city-in-state                             2467 samples
  family                                    506 samples
  gram1-adjective-to-adverb                 992 samples
  gram2-opposite                            812 samples
  gram3-comparative                         1332 samples
  gram4-superlative                         1122 samples
  gram5-present-participle                  1056 samples
  gram6-nationality-adjective               1599 samples
  gram7-past-tense                          1560 samples
  gram8-plural                              1332 samples
  gram9-plural-verbs                        870 samples


In [ ]:
# Evaluate accuracy on a few sections

def evaluate_section(section_name, samples, restrict_vocab=50000, max_test=100):
    correct = 0
    total   = 0

    for w1, w2, w3, w4 in samples[:max_test]:
        # Skip if any word is missing from vocabulary
        if not all(w in model for w in [w1, w2, w3, w4]):
            continue

        # Predict answer: vec(w2) - vec(w1) + vec(w3)
        prediction = model.most_similar(
            positive=[w2, w3],
            negative=[w1],
            topn=1,
            restrict_vocab=restrict_vocab   # <-- limits the search space
        )[0][0]

        if prediction == w4:
            correct += 1
        total += 1

    accuracy = (correct / total * 100) if total > 0 else 0
    return correct, total, accuracy


print(f'{"Section":<40} {"Correct":>8} {"Total":>7} {"Accuracy":>10}')
print('-' * 68)

for name in list(sections.keys())[:4]:   # test first 4 sections
    correct, total, acc = evaluate_section(name, sections[name])
    print(f'{name:<40} {correct:>8} {total:>7} {acc:>9.1f}%')

Section                                   Correct   Total   Accuracy
--------------------------------------------------------------------
capital-common-countries                        0      85       0.0%
capital-world                                   0      31       0.0%
currency                                        5      21      23.8%
city-in-state                                   0      98       0.0%


In [ ]:
# Show sample predictions — correct and incorrect

section_name = list(sections.keys())[2]
samples      = sections[section_name]

print(f'Section: {section_name}')
print(f'Formula: vec(w2) - vec(w1) + vec(w3) = predicted  (should match w4)')
print()
print(f'{"w1":<12} {"w2":<12} {"w3":<12} {"Expected":>12} {"Predicted":>12} {"OK?":>5}')
print('-' * 68)

shown = 0
for w1, w2, w3, w4 in samples:
    if not all(w in model for w in [w1, w2, w3, w4]):
        continue
    pred = model.most_similar(
        positive=[w2, w3], negative=[w1], topn=1, restrict_vocab=50000
    )[0][0]
    ok = 'YES' if pred == w4 else 'NO'
    print(f'{w1:<12} {w2:<12} {w3:<12} {w4:>12} {pred:>12} {ok:>5}')
    shown += 1
    if shown == 10:
        break

Section: currency
Formula: vec(w2) - vec(w1) + vec(w3) = predicted  (should match w4)

w1           w2           w3               Expected    Predicted   OK?
--------------------------------------------------------------------
argentina    peso         brazil               real        pesos    NO
argentina    peso         canada             dollar       dollar   YES
argentina    peso         croatia              kuna        pesos    NO
argentina    peso         denmark             krone        pesos    NO
argentina    peso         europe               euro        pesos    NO
argentina    peso         hungary            forint        pesos    NO
argentina    peso         india               rupee        rupee   YES
argentina    peso         iran                 rial       dollar    NO
argentina    peso         japan                 yen          yen   YES
argentina    peso         korea                 won        rupee    NO
